In [1]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from microgrid_old.techno_ka import techno_ka
from problems.benchmark_problems import get_pareto

from plotly.subplots import make_subplots

import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pickle
import plotly.colors as pc
import plotly.graph_objects as go

# 2D Plotting:

### MESH - 2D:

In [ ]:
design_space = []
pareto_front = []

objective_dim = 2
position_dim = 3
experiment_name = "LAG"

# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_dm_pool_type = [0] # [0, 1, 2]
choose_de_mutation_type = [0] # [0, 1, 2, 3, 4]

# Name of chosen files
file_names = [
        f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_dm_pool_type
        for k in choose_de_mutation_type
    ]
num_dataset = len(file_names) # Number of dataset

solid_colors = [mcolors.to_hex(cm.tab10(i)) for i in range(num_dataset + 1)] # List with solid colors

colorscales = pc.named_colorscales() # List of color scales
num_scales = len(colorscales)

for i in range(num_dataset):
  with open("../results/" + file_names[i], "rb") as f:
    r = pickle.load(f)
    design_space.append(
      go.Scatter3d(x=r['combined'][0][:,0],
                   y=r['combined'][0][:,1],
                   z=r['combined'][0][:,2],
                   mode='markers',
               	#    marker=dict(size=3, color=solid_colors[i]),
               	   marker=dict(size=3, color=r['combined'][1][:,0], colorscale=colorscales[(i+1) % num_scales]),
                   name='MESH - ' + file_names[i][5:11])
    )
    pareto_front.append(
      go.Scatter(x=r['combined'][1][:,0],
                 y=-r['combined'][1][:,1],
                 mode='markers',
             	#  marker=dict(size=3, color=solid_colors[i]),
             	 marker=dict(size=3, color=r['combined'][1][:,0], colorscale=colorscales[(i+1) % num_scales]),
                 name='MESH - ' + file_names[i][5:11],
                 showlegend=False)
    )

# Creating a subplot
fig = make_subplots(rows=1, cols=2, subplot_titles=("Pareto Front", "Design Space"), specs=[[{'type': 'scatter'}, {'type': 'scatter3d'}]])

for i in range(num_dataset):
    fig.add_trace(pareto_front[i], row=1, col=1)
    fig.add_trace(design_space[i], row=1, col=2)

# Setting layout
fig.update_layout(
    showlegend=True,
    # 2D layout
    xaxis=dict(
        title=dict(text='Price of Electricity [US$/kWh]', font=dict(size=14)),
    ),
    yaxis=dict(
        title=dict(text='Renewable Factor', font=dict(size=14)),
    ),
    # 3D layout
    scene=dict(
        camera=dict(
          eye=dict(x=1.5,y=1.5,z=1.5)
        ),
        xaxis=dict(
            title=dict(text='Max. PV Generation (x) [kWh]', font=dict(size=10)),
            backgroundcolor='white',  # X-axis background
            gridcolor='lightgray',    # Grid color on X axis 
            showbackground=True       # Show background
        ),
        yaxis=dict(
            title=dict(text='Max. WT Generation (y) [kWh]', font=dict(size=10)),
            backgroundcolor='white',  # Y-axis background
            gridcolor='lightgray',    # Grid color on Y axis
            showbackground=True       # Show background
        ),
        zaxis=dict(
            title=dict(text='Battery Capacity (z) [kWh]', font=dict(size=10)),
            backgroundcolor='white',  # Z-axis background
            gridcolor='lightgray',    # Grid color on Z axis
            showbackground=True       # Show background
        )
    ),
    title=dict(
        text='MESH on Microgrid',
        x=0.5,  # Centering the title
        xanchor='center',
        yanchor='top'
    ),
    #margin=dict(l=0, r=0, b=0, t=0), # Rearrange the margin
    width=1520,   # Figure width (pixels)
    height=720,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Showing the figure
fig.show()

### Plotting Pymoo vs MESH - 2D:

In [ ]:
# Creating a subplot
fig = make_subplots(rows=1, cols=2, subplot_titles=("Pareto Front", "Design Space"), specs=[[{'type': 'scatter'}, {'type': 'scatter3d'}]])

fig.add_trace(go.Scatter(x=res.F[:,0], 
               y=res.F[:,1],
               mode='markers', marker=dict(size=5, color=solid_colors[-1]),
               name="NSGA-II"
              ), row=1, col=1)
fig.add_trace(go.Scatter3d(x=res.X[:,0], 
               y=res.X[:,1],
               z=res.X[:,2],#[150]*len(res.F['combined'][0][:,0]),
               mode='markers', marker=dict(size=3, color=solid_colors[-1]),
               name="NSGA-II",
               showlegend=False
              ), row=1, col=2)

add_MESH = True
if(add_MESH):
    for i in range(num_dataset):
        fig.add_trace(pareto_front[i], row=1, col=1)
        fig.add_trace(design_space[i], row=1, col=2)

# Setting layout
fig.update_layout(
    showlegend=True,
    # 2D layout
    xaxis=dict(
        title=dict(text='Planning Cost [US$]', font=dict(size=14)),
        #title=dict(text='Price of Electricity [€$/kWh]', font=dict(size=14)),
    ),
    yaxis=dict(
        title=dict(text='Renewable Factor', font=dict(size=14)),
        #title=dict(text='Price of Electricity [€$/kWh]', font=dict(size=14)),
        #title=dict(text='(-1) Renewable Factor', font=dict(size=14)),
    ),
    scene=dict(
        camera=dict(
          eye=dict(x=1.5,y=1.5,z=1.5)
        ),
        xaxis=dict(
            title=dict(text='Max. PV Generation (x) [kWh]', font=dict(size=10)),
            backgroundcolor='white',  # X-axis background
            gridcolor='lightgray',    # Grid color on X axis 
            showbackground=True       # Show background
        ),
        yaxis=dict(
            title=dict(text='Number of Turbines (y)', font=dict(size=10)),
            backgroundcolor='white',  # Y-axis background
            gridcolor='lightgray',    # Grid color on Y axis
            showbackground=True       # Show background
        ),
        zaxis=dict(
            title=dict(text='Battery Capacity (z) [kWh]', font=dict(size=10)),
            backgroundcolor='white',  # Z-axis background
            gridcolor='lightgray',    # Grid color on Z axis
            showbackground=True       # Show background
        )
    ),
    title=dict(
        text='MESH vs NSGA-II',
        x=0.5,  # Centering the title
        xanchor='center',
        yanchor='top'
    ),
    #margin=dict(l=0, r=0, b=0, t=0), # Rearrange the margin
    width=1520,   # Figure width (pixels)
    height=720,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Showing the figure
fig.show()

# 3D Plotting:

In [ ]:
num_final_solutions = 300
position_dim = 3 # Number of variables
design_space = []
pareto_front = []

 # LAG AGM(0) Li4Ti5O12(1) LiCoO2(2) LiFePO4(3) LiMnO2(4) LiNiCoMnO2(5) LiNiCoAlO2(6) LiPoly(7) NaNiCl(8) NaS(9) NiCd(10) NiMH(11) RFV(12) Zn/Br Redox(13)
select_bat = 3
bat_name = ['LAG', 'LTO', 'LCO', 'LFP', 'LMO', 'LNCMO', 'LNCAO', 'LPoly', 'NNC', 'NaS', 'NiC', 'NMH', 'RFV', 'ZnBr']
experiment_name = bat_name[select_bat]

# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2 ,3]
choose_dm_pool_type = [1] # [0, 1, 2]
choose_de_mutation_type = [0] # [0, 1, 2, 3, 4]

# Name of chosen files
file_names = [
        f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}.pkl"
        for i in choose_global_best_attribution_type
        for j in choose_dm_pool_type
        for k in choose_de_mutation_type
    ]
num_dataset = len(file_names) # Number of dataset

##### Color setup #####
solid_colors = [mcolors.to_hex(cm.tab10(i+1)) for i in range(num_dataset)] # List with solid colors
colorscales = pc.named_colorscales() # List of color scales
num_scales = len(colorscales)
#######################

##### Marker setup #####
marker_symbols = ['circle', 'circle-open', 'diamond', 'diamond-open', 'square', 'square-open', 'x']
num_symbols = len(marker_symbols)
#######################

for i in range(num_dataset):
  with open("../scripts/results/" + file_names[i], "rb") as f:
    r = pickle.load(f)
    design_space.append(
      go.Scatter3d(x=r['combined'][0][:,0], 
                   y=r['combined'][0][:,1],
                   z=r['combined'][0][:,2], #[150]*len(r['combined'][0][:,0]),
                   mode='markers', marker=dict(size=3, color=solid_colors[i]), #r['combined'][1][:,1], colorscale=colorscales[(i+1) % num_scales]),
                   name='MESH - ' + file_names[i][:6])
    )
    pareto_front.append(
      go.Scatter3d(x=r['combined'][1][:,0],
                   y=r['combined'][1][:,1],
                   z=-r['combined'][1][:,2],
                   mode='markers', marker=dict(size=3, color=solid_colors[i]), #color=r['combined'][1][:,1], colorscale=colorscales[(i+1) % num_scales]),
                   name='MESH - ' + file_names[i][:8],
                   showlegend=False)
    )

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]])

for i in range(num_dataset):
    fig.add_trace(pareto_front[i], row=1, col=1)
    fig.add_trace(design_space[i], row=1, col=2)

axis_range = None

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"
        )
    ],
    annotations=[
        dict(text="Non-dominated Front", x=0.23, y=0.85, showarrow=False, font=dict(size=14)),
        dict(text="Design Space", x=0.78, y=0.85, showarrow=False, font=dict(size=14))
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=2,y=2,z=2)
        ),
        xaxis=dict(
            title=dict(text='LOLP (x)', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # X-axis background
            gridcolor='lightgray',     # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='Price of Electricity (y) [€$/kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Y-axis background
            gridcolor='lightgray',     # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='Renewable Factor (z)', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Z-axis background
            gridcolor='lightgray',     # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        aspectmode='cube'
    ),
    scene2=dict(
        camera=dict(
          eye=dict(x=2,y=2,z=2)
        ),
        xaxis=dict(
            title=dict(text='Max. PV Generation (x) [kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # X-axis background
            gridcolor='lightgray',     # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='Number of Turbines (y)', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Y-axis background
            gridcolor='lightgray',     # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='Battery Capacity (z) [kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Z-axis background
            gridcolor='lightgray',     # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        )
    ),
    legend=dict(
        x=0.99,
        y=0.95,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     y=0.95,
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=5, r=5, b=5, t=5), # Rearrange the margin
    width=1200,   # Figure width (pixels)
    height=700,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

fig.show()

### Pymoo - 3D:

In [ ]:
num_run = 10 # Number of runs
population_size = 100 # Population size
max_fit_eval = 6000 # Maximum number of fitness evaluations

prob_rec = 0.8
eta_rec = 15
prob_mut = 0.2
eta_mut = 9.7
crossover = SBX(prob=prob_rec, prob_var=1.0, eta=eta_rec)
mutation = PolynomialMutation(prob=prob_mut, eta=eta_mut)

algorithm = NSGA2(
    pop_size=population_size,
    crossover=crossover,
    mutation=mutation,
    eliminate_duplicates=True
)

solar_data = np.genfromtxt('../scripts/microgrid_old/seasonal_data/solreal.txt')
wind_data = np.genfromtxt('../scripts/microgrid_old/seasonal_data/wind_data.txt')
load_ind = np.genfromtxt('../scripts/microgrid_old/seasonal_data/loadind.txt')
xl = np.array([10, 1, 50]) # Lower bound of problem [max PV generation, number of wind turbines, battery capacity]
xu = np.array([450, 5, 500]) # Upper bound of problem [max PV generation, number of wind turbines, battery capacity]
def func(args):
    r = techno_ka(args[0], args[1], 0.8, args[2], select_bat, solar_data, wind_data, load_ind)[:3]
    r[-1] = -r[-1] # Maximizing renewable factor
    return r

class MyProblem(Problem):
    def __init__(self, n_var, xl, xu):
        super().__init__(n_var=n_var, n_obj=3, n_constr=0, xl=xl, xu=xu)

    def _evaluate(self, X, out, *args, **kwargs):
        out["F"] = np.array([func(x) for x in X])

problem = MyProblem(n_var=position_dim, xl=xl, xu=xu)

nsga2_results = {}
combined_F = None
combined_P = None
for i in range(num_run):
  res = minimize(problem,
                algorithm,
                ('n_eval', max_fit_eval),
                verbose=False)

  Pos, Fit = res.X, res.F
  nsga2_results[i+1] = {"F":Fit, "P":Pos}
  # Accumulates the results of all executions
  if combined_F is None:
    combined_P = Pos
    combined_F = Fit
  else:
    combined_P = np.vstack((combined_P, Pos))
    combined_F = np.vstack((combined_F, Fit))

# Getting the unique points
unique_combined_P, unique_idxs = np.unique(combined_P, axis=0, return_index=True)
unique_combined_F = combined_F[unique_idxs]
# Sorting the vector Fit
# Return: (non dominated front, domination list, domination counter, non domination ranks)
if len(unique_combined_F) == 1:
    ndf = [[0]]
else:
    ndf, _, _, _ = fast_non_dominated_sorting(points=unique_combined_F)
n = min(len(ndf[0]), num_final_solutions)
# Get the best indexes based on number of final solutions
nsga2_pareto_front = unique_combined_F[ndf[0]]
best_idx = select_best_N_mo(nsga2_pareto_front, n)
nsga2_results['combined'] = (unique_combined_P[ndf[0]][best_idx], nsga2_pareto_front[best_idx])
with open(f'../result/NSGA2_{experiment_name}.pkl', 'wb') as file:
  pickle.dump(nsga2_results, file)

### Plotting Pymoo vs MESH - 3D:

In [ ]:
nsga2_result = pickle.load(open(f'../result/NSGA2_{experiment_name}.pkl', 'rb'))

# Creating a subplot
fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]])

fig.add_trace(go.Scatter3d(x=nsga2_result['combined'][1][:,0], 
               y=nsga2_result['combined'][1][:,1],
               z=-nsga2_result['combined'][1][:,2],
               mode='markers', marker=dict(size=5, symbol='cross', color=mcolors.to_hex(cm.tab10(0))),
               name="NSGA-II"),
               row=1, col=1
               )
fig.add_trace(go.Scatter3d(x=nsga2_result['combined'][0][:,0], 
               y=nsga2_result['combined'][0][:,1],
               z=nsga2_result['combined'][0][:,2],
               mode='markers', marker=dict(size=5, symbol='cross', color=mcolors.to_hex(cm.tab10(0))),
               name="NSGA-II",
               showlegend=False),
               row=1, col=2
               )

add_MESH = True
if(add_MESH):
    for i in range(num_dataset):
        fig.add_trace(pareto_front[i], row=1, col=1)
        fig.add_trace(design_space[i], row=1, col=2)

axis_range = None

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"
        )
    ],
    annotations=[
        dict(text="Non-dominated Front", x=0.23, y=0.85, showarrow=False, font=dict(size=14)),
        dict(text="Design Space", x=0.78, y=0.85, showarrow=False, font=dict(size=14))
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=2,y=2,z=2)
        ),
        xaxis=dict(
            title=dict(text='LOLP (x)', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # X-axis background
            gridcolor='lightgray',     # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='Price of Electricity (y) [€$/kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Y-axis background
            gridcolor='lightgray',     # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='Renewable Factor (z)', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Z-axis background
            gridcolor='lightgray',     # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        aspectmode='cube'
    ),
    scene2=dict(
        camera=dict(
          eye=dict(x=2,y=2,z=2)
        ),
        xaxis=dict(
            title=dict(text='Max. PV Generation (x) [kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # X-axis background
            gridcolor='lightgray',     # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='Number of Turbines (y)', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Y-axis background
            gridcolor='lightgray',     # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='Battery Capacity (z) [kWh]', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',   # Z-axis background
            gridcolor='lightgray',     # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        )
    ),
    legend=dict(
        x=0.99,
        y=0.95,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=5, r=5, b=5, t=5), # Rearrange the margin
    width=1200,   # Figure width (pixels)
    height=700,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Showing the figure
fig.show()

# Benchmark 2D:

In [ ]:
objective_dim = 2 # Number of objectives
position_dim = 10 # Number of variables
n_pareto_points = 5000 # Number of Pareto points to be generated
wfg_k = 1 # The parameter k for WFG problems

experiment_name = "zdt6"

# Choosing the files
insert_old_mesh = True
insert_nsga2 = True

# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_dm_pool_type = [0] # [0, 1, 2]
choose_de_mutation_type = [0] # [0, 1, 2, 3, 4]

# Name of chosen files
file_tuple = [
        (f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1}')
        for i in choose_global_best_attribution_type
        for j in choose_dm_pool_type
        for k in choose_de_mutation_type
    ]

if insert_old_mesh:
	old_global_best_attribution_type = 1 # [0, 1, 2, 3]
	old_dm_pool_type = 0 # [0, 1, 2]
	old_de_mutation_type = 0 # [0, 1, 2, 3, 4]
	file_tuple.append((f'OLD_MESH_G{old_global_best_attribution_type+1}S{old_dm_pool_type+1}D{old_de_mutation_type+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'MESH - ' + f'G{old_global_best_attribution_type+1}S{old_dm_pool_type+1}D{old_de_mutation_type+1} (Orig.)'))
if insert_nsga2:
  	file_tuple.append((f'NSGA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSGA2'))

num_datasets = len(file_tuple) # Number of dataset

##### Color setup #####
solid_colors = [mcolors.to_hex(cm.tab10(i)) for i in range(num_datasets)] # List with solid colors
colorscales = pc.named_colorscales() # List of color scales
num_scales = len(colorscales)
#######################

##### Marker setup #####
marker_symbols = ['circle', 'diamond', 'square', 'cross', 'x', 'circle-open', 'diamond-open', 'square-open']
num_symbols = len(marker_symbols)
#######################

pareto_front = []
for i in range(num_datasets):
  with open("../results/" + file_tuple[i][0], "rb") as f:
    r = pickle.load(f)
    pareto_front.append(
      go.Scatter(x=r['combined'][1][:,0],
                   y=r['combined'][1][:,1],
                   mode='markers', 
                   marker=dict(size=8, symbol=marker_symbols[i % num_symbols], color=solid_colors[i]), #r['combined'][1][:,1], colorscale=colorscales[(i+1) % num_scales]),
                   name=file_tuple[i][1],
                   showlegend=True)
    )

## Plotting

In [33]:
# Creating a subplot
fig = go.Figure()

for i in range(num_datasets):
    fig.add_trace(pareto_front[i])

axis_range = None # [-0.05, 1.05]
# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"  # Transparente
        )
    ],
    xaxis=dict(title=dict(text='f1', font=dict(size=14)), tickfont=dict(size=14)),
    yaxis=dict(title=dict(text='f2', font=dict(size=14)), tickfont=dict(size=14)),
    # title=dict(
    #     text='MESH - ' + experiment_name.upper(),
    #     x=0.5,  # Centering the title
    #     xanchor='center',
    #     yanchor='top'
    # ),
    legend=dict(
        x=0.99,
        y=0.99,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",  # fundo leve para não colidir com os pontos
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    ),
    showlegend=True,
    margin=dict(l=5, r=5, b=5, t=5), # Rearrange the margin
    width=600,   # Figure width (pixels)
    height=480,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Plotting the Pareto front
real_pareto_front = get_pareto(experiment_name, n_pareto_points, position_dim, 2, wfg_k)
fig.add_trace(
	go.Scatter(x=real_pareto_front[:,0],
		y=real_pareto_front[:,1],
		mode='markers', marker=dict(size=2, color='black'),
		name='Pareto Front',
		showlegend=True
		)
	)

# Showing the figure
fig.show()

# Benchmark 3D:

In [ ]:
objective_dim = 3 # Number of objectives
position_dim = 10 # Number of variables
n_pareto_density = 50 # Number of Pareto points to be generated
wfg_k = 6 # The parameter k for WFG problems

experiment_name = "dtlz3"

# Choosing the files
insert_old_mesh = True
insert_nsga2 = True

# Choosing the files
choose_global_best_attribution_type = [0] # [0, 1, 2, 3]
choose_dm_pool_type = [1] # [0, 1, 2]
choose_de_mutation_type = [0] # [0, 1, 2, 3, 4]

# Name of chosen files
file_tuple = [
        (f"MESH_G{i+1}S{j+1}D{k+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl", 'MESH - ' + f'G{i+1}S{j+1}D{k+1}')
        for i in choose_global_best_attribution_type
        for j in choose_dm_pool_type
        for k in choose_de_mutation_type
    ]

if insert_old_mesh:
	old_global_best_attribution_type = 1 # [0, 1, 2, 3]
	old_dm_pool_type = 1 # [0, 1, 2]
	old_de_mutation_type = 0 # [0, 1, 2, 3, 4]
	file_tuple.append((f'OLD_MESH_G{old_global_best_attribution_type+1}S{old_dm_pool_type+1}D{old_de_mutation_type+1}_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'MESH - ' + f'G{old_global_best_attribution_type+1}S{old_dm_pool_type+1}D{old_de_mutation_type+1} (Orig.)'))
if insert_nsga2:
  	file_tuple.append((f'NSGA2_{experiment_name}_{objective_dim}_{position_dim}.pkl', 'NSGA2'))

num_datasets = len(file_tuple) # Number of dataset

##### Color setup #####
solid_colors = [mcolors.to_hex(cm.tab10(i)) for i in range(num_datasets)] # List with solid colors
colorscales = pc.named_colorscales() # List of color scales
num_scales = len(colorscales)
#######################

##### Marker setup #####
marker_symbols = ['circle', 'diamond', 'square', 'cross', 'x', 'circle-open', 'diamond-open', 'square-open']
num_symbols = len(marker_symbols)
#######################

pareto_front = []
for i in range(num_datasets):
  with open("../results/" + file_tuple[i][0], "rb") as f:
    r = pickle.load(f)
    pareto_front.append(
      go.Scatter3d(x=r['combined'][1][:,0],
                   y=r['combined'][1][:,1],
                   z=r['combined'][1][:,2],
                   mode='markers', marker=dict(size=5, symbol=marker_symbols[i % num_symbols], color=solid_colors[i]), #r['combined'][1][:,1], colorscale=colorscales[(i+1) % num_scales]),
                   name=file_tuple[i][1],
                   showlegend=True)
    )

## Plotting

In [10]:
# Creating a subplot
fig = go.Figure()

for i in range(num_datasets):
    fig.add_trace(pareto_front[i])

axis_range = None # [-0.1, 1.2]

# Setting layout
fig.update_layout(
    shapes=[
        dict(
            type="rect",
            xref="paper", yref="paper",
            x0=0, y0=0, x1=1, y1=1,
            line=dict(color="black", width=2),
            fillcolor="rgba(0,0,0,0)"  # Transparente
        )
    ],
    scene=dict(
        camera=dict(
          eye=dict(x=1,y=1,z=1)
        ),
        xaxis=dict(
            title=dict(text='f1', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # X-axis background
            gridcolor='lightgray',    # Grid color on X axis 
            showbackground=True,       # Show background
            range=axis_range
        ),
        yaxis=dict(
            title=dict(text='f2', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # Y-axis background
            gridcolor='lightgray',    # Grid color on Y axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        zaxis=dict(
            title=dict(text='f3', font=dict(size=14)),
            tickfont=dict(size=12),
            backgroundcolor='white',  # Z-axis background
            gridcolor='lightgray',    # Grid color on Z axis
            showbackground=True,       # Show background
            range=axis_range
        ),
        aspectmode='cube'
    ),
    legend=dict(
        x=0.99,
        y=0.99,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.8)",  # fundo leve para não colidir com os pontos
        bordercolor="black",
        borderwidth=1,
        font=dict(size=14)
    ),
    showlegend=True,
    # title=dict(
    #     text='MESH - ' + experiment_name,
    #     x=0.5,  # Centering the title
    #     xanchor='center',
    #     yanchor='top'
    # ),
    margin=dict(l=5, r=5, b=5, t=5), # Rearrange the margin
    width=600,   # Figure width (pixels)
    height=420,  # Figure height (pixels)
    paper_bgcolor="white",
    plot_bgcolor='white'
)

# Plotting the Pareto front
real_pareto_front = get_pareto(experiment_name, n_pareto_density, position_dim, 3, wfg_k)
fig.add_trace(
	go.Scatter3d(x=real_pareto_front[:,0],
		y=real_pareto_front[:,1],
		z=real_pareto_front[:,2],
		mode='markers', marker=dict(size=2, color='black'),
		name='Pareto Front',
		showlegend=True
		)
	)

# Showing the figure
fig.show()